# Order Issue Handler

A simple tool-calling application built with Python, the Google GenAI SDK, and Gradio to automate e-commerce order exception handling.

## Problem Statement
When a customer reports a delayed or damaged order, support teams normally check three separate systems—order status, warehouse inventory, and ticketing—before deciding whether to reship or refund. This project automates that workflow using three Python tools and logs every run to a CSV file.

## Tools Used
1. **`get_order(order_id)`** — Looks up order details, item code, delivery status, and whether the order is past due.
2. **`check_stock(item)`** — Checks available inventory across Mumbai, Hyderabad, and Bengaluru warehouses.
3. **`log_action(order_id, action, note)`** — Records the final action (`reship`, `refund`, or `escalate`) and returns a ticket ID.

## Project Files
- `order_agent.ipynb` (or `.py`) — Main script with the 3 tools, decision rules, CSV logger, and Gradio UI.
- `order_agent_output.csv` — Automatically generated log of all queries, tool calls, and final replies.
- `README.md` — Project documentation.

## Setup & How to Run

1. Install the required packages:
   ```bash
   pip install google-genai gradio


In [ ]:
!pip install -q google-genai

In [ ]:
import os
from google import genai
from google.genai import types
from google.colab import userdata

# Retrieve the Gemini API key from Colab Secrets
try:
    # Key names cannot have spaces (e.g., 'GEMINI_API_KEY')
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    client = genai.Client()
except Exception:
    # Wrap the fallback API key string in quotes to avoid a NameError
    fallback_key = "AQ.Ab8RN6JkCAB5fz9QLwe97AirGUh_ihLvMA48uc4eA26W3V7gvQ"
    client = genai.Client(api_key=fallback_key)
    import notebook
    print("Using fallback API key. Please set your 'GEMINI_API_KEY' in the Colab Secrets tab without spaces.")

# sample data
orders = {
    "ORD-101": {
        "customer": "Rahul Verma",
        "item": "LAPTOP-15",
        "status": "Delayed in transit",
        "past_due": True,
        "price": 72000,
    },
    "ORD-102": {
        "customer": "Sneha Reddy",
        "item": "WATCH-PRO",
        "status": "Damaged on delivery",
        "past_due": True,
        "price": 12500,
    },
}

inventory = {
    "LAPTOP-15": {"Mumbai": 4, "Hyderabad": 2, "Bengaluru": 0},
    "WATCH-PRO": {"Mumbai": 0, "Hyderabad": 0, "Bengaluru": 0},
}


def get_order(order_id: str) -> dict:
    """Look up order details by order_id."""
    print(f"-> calling get_order({order_id})")
    data = orders.get(order_id.upper())
    if not data:
        return {"found": False, "message": "Order not found"}
    return {"found": True, "order_id": order_id.upper(), **data}


def check_stock(item: str) -> dict:
    """Check how many units of an item are left across warehouses."""
    print(f"-> calling check_stock({item})")
    warehouses = inventory.get(item.upper())
    if warehouses is None:
        return {"found": False, "total": 0, "warehouses": {}}
    return {
        "found": True,
        "item": item.upper(),
        "total": sum(warehouses.values()),
        "warehouses": warehouses,
    }


def log_action(order_id: str, action: str, note: str) -> dict:
    """Save the final action (reship, refund, or escalate) and return a ticket number."""
    print(f"-> calling log_action({order_id}, {action})")
    return {
        "status": "done",
        "ticket": f"TKT-{order_id.upper()}-01",
        "action": action,
        "note": note,
    }


rules = """
You handle order issues for a store.
1. First call get_order using the order ID. If it is not found, just tell the user.
2. If past_due is True, call check_stock for that item.
3. If total stock is more than 0, call log_action with action='reship' and mention which city has stock.
4. If total stock is 0, call log_action with action='refund'.
Keep the final reply short and clear with the ticket number.
"""


def run(query: str):
    print("\nQuery:", query)
    res = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=query,
        config=types.GenerateContentConfig(
            system_instruction=rules,
            tools=[get_order, check_stock, log_action],
        ),
    )
    print("\nReply:\n" + res.text)
    print("-" * 50)


# test cases
try:
    run("Order ORD-101 is late, what should we do?")
    run("Customer got a broken item for ORD-102, please handle it.")
    run("Check order ORD-999")
except Exception as e:
    print(f"Error executing requests: {e}")


Query: Order ORD-101 is late, what should we do?
-> calling get_order(ORD-101)
-> calling check_stock(LAPTOP-15)
-> calling log_action(ORD-101, reship)

Reply:
Order ORD-101 is past due. Stock is available in Hyderabad and Mumbai. The order has been reshipped. Ticket number: TKT-ORD-101-01.
--------------------------------------------------

Query: Customer got a broken item for ORD-102, please handle it.
-> calling get_order(ORD-102)
-> calling check_stock(WATCH-PRO)
-> calling log_action(ORD-102, refund)

Reply:
The item was out of stock across all warehouses, so a refund has been issued. Ticket number: TKT-ORD-102-01.
--------------------------------------------------

Query: Check order ORD-999
-> calling get_order(ORD-999)

Reply:
Order ORD-999 was not found in the system.
--------------------------------------------------


In [9]:
import csv

# track tool calls per run
current_tools = []
output_rows = []


def get_order(order_id: str) -> dict:
    """Look up order details by order_id."""
    current_tools.append(f"get_order({order_id})")
    print(f"-> calling get_order({order_id})")
    data = orders.get(order_id.upper())
    if not data:
        return {"found": False, "message": "Order not found"}
    return {"found": True, "order_id": order_id.upper(), **data}


def check_stock(item: str) -> dict:
    """Check how many units of an item are left across warehouses."""
    current_tools.append(f"check_stock({item})")
    print(f"-> calling check_stock({item})")
    warehouses = inventory.get(item.upper())
    if warehouses is None:
        return {"found": False, "total": 0, "warehouses": {}}
    return {
        "found": True,
        "item": item.upper(),
        "total": sum(warehouses.values()),
        "warehouses": warehouses,
    }


def log_action(order_id: str, action: str, note: str) -> dict:
    """Save the final action (reship, refund, or escalate) and return a ticket number."""
    current_tools.append(f"log_action({order_id}, {action})")
    print(f"-> calling log_action({order_id}, {action})")
    return {
        "status": "done",
        "ticket": f"TKT-{order_id.upper()}-01",
        "action": action,
        "note": note,
    }


def run(query: str):
    current_tools.clear()
    print("\nQuery:", query)
    res = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=query,
        config=types.GenerateContentConfig(
            system_instruction=rules,
            tools=[get_order, check_stock, log_action],
        ),
    )
    reply = res.text.strip()
    print("\nReply:\n" + reply)
    print("-" * 50)

    output_rows.append({
        "query": query,
        "tools_called": " -> ".join(current_tools),
        "final_reply": reply,
    })


# test cases
run("Order ORD-101 is late, what should we do?")
run("Customer got a broken item for ORD-102, please handle it.")
run("Check order ORD-999")

# export to CSV
with open("order_agent_output.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["query", "tools_called", "final_reply"])
    writer.writeheader()
    writer.writerows(output_rows)

print("Saved results to order_agent_output.csv")



Query: Order ORD-101 is late, what should we do?
-> calling get_order(ORD-101)
-> calling check_stock(LAPTOP-15)
-> calling log_action(ORD-101, reship)

Reply:
Order ORD-101 has been reshipped. Stock is available in Mumbai and Hyderabad. Ticket number: TKT-ORD-101-01.
--------------------------------------------------

Query: Customer got a broken item for ORD-102, please handle it.
-> calling get_order(ORD-102)
-> calling check_stock(WATCH-PRO)
-> calling log_action(ORD-102, refund)

Reply:
The item was out of stock across all warehouses, so a refund has been issued. 
Ticket Number: TKT-ORD-102-01
--------------------------------------------------

Query: Check order ORD-999
-> calling get_order(ORD-999)

Reply:
Order ORD-999 was not found. Please check the order ID and try again.
--------------------------------------------------
Saved results to order_agent_output.csv


In [10]:
!pip install -q gradio


In [11]:
import os
import csv
import gradio as gr
from google.genai import types

CSV_FILE = "order_agent_output.csv"


def handle_query(query: str):
    if not query.strip():
        return "No tools called", "Please enter an order query."

    current_tools.clear()
    res = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=query,
        config=types.GenerateContentConfig(
            system_instruction=rules,
            tools=[get_order, check_stock, log_action],
        ),
    )

    trace = "\n".join(f"{i+1}. {t}" for i, t in enumerate(current_tools)) or "None"
    reply = res.text.strip()

    # log every UI test to CSV
    file_exists = os.path.isfile(CSV_FILE)
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["query", "tools_called", "final_reply"])
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            "query": query,
            "tools_called": " -> ".join(current_tools),
            "final_reply": reply,
        })

    return trace, reply


demo = gr.Interface(
    fn=handle_query,
    inputs=gr.Textbox(
        label="Order Issue",
        placeholder="e.g., Order ORD-101 is late, what should we do?",
        lines=2,
    ),
    outputs=[
        gr.Textbox(label="Tools Called", lines=4),
        gr.Textbox(label="Final Reply", lines=4),
    ],
    title="Order Issue Handler",
    description="Enter an order ID issue below or click one of the test cases.",
    examples=[
        ["Order ORD-101 is late, what should we do?"],
        ["Customer got a broken item for ORD-102, please handle it."],
        ["Check order ORD-999"],
    ],
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://43ec9d574997feb9ec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
